
# 03c — ACE production training and candidate selection

This notebook prepares and launches **four ACE candidate models** on the **Zenodo Cu–Zr dataset**.

Project logic:
- **This notebook does not make the final scientific model choice.**
- It creates a small, controlled set of ACE candidates.
- The real production model will be chosen later using **Notebook 05** validation.

We keep **four ACE models** for symmetry with the four MACE models.



## Design philosophy

We use a **controlled candidate sweep**, not broad hyperparameter optimization.

The main idea is:
- keep the dataset split fixed,
- keep the training workflow fixed,
- vary mainly the **ACE basis complexity**,
- use the same **7.6 Å cutoff** for the primary sweep.

This follows the logic of the ACE Cu–Zr paper: larger ACE models improved training accuracy but could reduce transferability and increase cost, so the smallest model was preferred for production use.


In [ ]:

# Core imports
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)



## User configuration

Adjust the paths below before running on the target machine.

Notes:
- `train.extxyz` is the main training pool.
- `test.extxyz` stays untouched.
- We create `train_split.extxyz` and `valid_split.extxyz` from `train.extxyz`.
- Configs, scripts, logs, and models are written under `ACE_RUN_ROOT`.


In [ ]:

# ---- User paths ----
PROJECT_ROOT = Path("../cu_zr_mlip_project").resolve()
DATA_ROOT = PROJECT_ROOT / "datasets"
TRAIN_FILE = DATA_ROOT / "train.extxyz"
TEST_FILE = DATA_ROOT / "test.extxyz"

ACE_RUN_ROOT = PROJECT_ROOT / "ace_runs"
ACE_CONFIG_ROOT = ACE_RUN_ROOT / "configs"
ACE_SCRIPT_ROOT = ACE_RUN_ROOT / "scripts"
ACE_LOG_ROOT = ACE_RUN_ROOT / "logs"
ACE_MODEL_ROOT = ACE_RUN_ROOT / "models"
ACE_SPLIT_ROOT = ACE_RUN_ROOT / "splits"
ACE_SUMMARY_FILE = ACE_RUN_ROOT / "ace_candidate_summary.csv"

VALID_FRAC = 0.10
FORCE_NEW_SPLIT = False

for p in [ACE_RUN_ROOT, ACE_CONFIG_ROOT, ACE_SCRIPT_ROOT, ACE_LOG_ROOT, ACE_MODEL_ROOT, ACE_SPLIT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_FILE:", TRAIN_FILE)
print("TEST_FILE:", TEST_FILE)
print("ACE_RUN_ROOT:", ACE_RUN_ROOT)



## ACE training backend

This notebook is written as a **project-ready template**.

You should set the actual ACE training command in one place below.
Examples of possible backends:
- a Python CLI from an ACE package,
- a Julia/ACEpotentials workflow,
- a custom shell launcher,
- a cluster submission wrapper.

The notebook generates:
- candidate metadata,
- config files,
- launch scripts,
- summary tables.

Then you can run locally or on the GPU machine.


In [ ]:

# ---- ACE training command template ----
# Replace this with the real training command once the ACE environment is finalized.
#
# The placeholders available are:
#   {config_file}
#   {log_file}
#   {model_dir}
#   {train_file}
#   {valid_file}
#   {test_file}
#   {candidate}
#
# Example ideas:
# ACE_TRAIN_CMD_TEMPLATE = "python train_ace.py --config {config_file} > {log_file} 2>&1"
# ACE_TRAIN_CMD_TEMPLATE = "julia run_ace_fit.jl {config_file} > {log_file} 2>&1"
# ACE_TRAIN_CMD_TEMPLATE = "bash launch_ace_train.sh {config_file} {log_file}"

ACE_TRAIN_CMD_TEMPLATE = None

# Optional: if your backend writes a final metrics JSON, point to its expected filename pattern.
METRICS_FILENAME = "metrics.json"
MODEL_FILENAME_HINTS = ["model.json", "model.yaml", "potential.json", "ace_model.json"]



## Candidate definitions

We keep **four ACE candidates** for symmetry with the four MACE candidates.

Primary sweep:
- fixed cutoff = **7.6 Å**
- vary basis/model complexity

Suggested interpretation:
- **ACE_A**: smallest / fastest / likely best transferability
- **ACE_B**: medium-small
- **ACE_C**: medium-large
- **ACE_D**: largest / stress-test for overfitting and runtime


In [ ]:

@dataclass
class ACECandidate:
    name: str
    cutoff: float
    basis_functions: int
    max_body_order: Optional[int] = None
    max_degree: Optional[int] = None
    notes: str = ""

ACE_CANDIDATES: List[ACECandidate] = [
    ACECandidate("ACE_A", cutoff=7.6, basis_functions=514,  notes="small / likely strongest transferability"),
    ACECandidate("ACE_B", cutoff=7.6, basis_functions=1352, notes="medium-small"),
    ACECandidate("ACE_C", cutoff=7.6, basis_functions=2838, notes="medium-large"),
    ACECandidate("ACE_D", cutoff=7.6, basis_functions=6084, notes="largest / most expensive"),
]

pd.DataFrame([asdict(c) for c in ACE_CANDIDATES])



## Dataset split helpers

We split only `train.extxyz` into:
- `train_split.extxyz`
- `valid_split.extxyz`

The held-out `test.extxyz` stays untouched.


In [ ]:

from ase.io import read, write

def read_extxyz_frames(path: Path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return read(path.as_posix(), index=":")


def make_train_valid_split(
    train_file: Path,
    out_train: Path,
    out_valid: Path,
    valid_frac: float = 0.10,
    seed: int = 42,
    force: bool = False,
):
    out_train = Path(out_train)
    out_valid = Path(out_valid)

    if out_train.exists() and out_valid.exists() and not force:
        print("Using existing split files:")
        print(" ", out_train)
        print(" ", out_valid)
        return out_train, out_valid

    frames = read_extxyz_frames(train_file)
    n = len(frames)
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_valid = max(1, int(round(valid_frac * n)))
    valid_idx = np.sort(idx[:n_valid])
    train_idx = np.sort(idx[n_valid:])

    train_frames = [frames[i] for i in train_idx]
    valid_frames = [frames[i] for i in valid_idx]

    write(out_train.as_posix(), train_frames, format="extxyz")
    write(out_valid.as_posix(), valid_frames, format="extxyz")

    print(f"Total frames: {n}")
    print(f"Train frames: {len(train_frames)}")
    print(f"Valid frames: {len(valid_frames)}")
    print("Wrote:")
    print(" ", out_train)
    print(" ", out_valid)
    return out_train, out_valid


In [ ]:

TRAIN_SPLIT_FILE = ACE_SPLIT_ROOT / "train_split.extxyz"
VALID_SPLIT_FILE = ACE_SPLIT_ROOT / "valid_split.extxyz"

TRAIN_SPLIT_FILE, VALID_SPLIT_FILE = make_train_valid_split(
    TRAIN_FILE,
    TRAIN_SPLIT_FILE,
    VALID_SPLIT_FILE,
    valid_frac=VALID_FRAC,
    seed=SEED,
    force=FORCE_NEW_SPLIT,
)



## Candidate config generation

The exact schema below is intentionally simple and backend-agnostic.

You can either:
- use it directly if your ACE backend reads JSON/YAML configs,
- or treat it as project metadata and translate it into the final backend input format.


In [ ]:

def candidate_to_config(candidate: ACECandidate) -> Dict[str, Any]:
    return {
        "candidate_name": candidate.name,
        "dataset": {
            "train_file": str(TRAIN_SPLIT_FILE),
            "valid_file": str(VALID_SPLIT_FILE),
            "test_file": str(TEST_FILE),
        },
        "model": {
            "architecture": "ACE",
            "cutoff_A": candidate.cutoff,
            "basis_functions": candidate.basis_functions,
            "max_body_order": candidate.max_body_order,
            "max_degree": candidate.max_degree,
        },
        "training": {
            "seed": SEED,
            "notes": candidate.notes,
        },
        "outputs": {
            "run_root": str(ACE_RUN_ROOT),
            "model_dir": str(ACE_MODEL_ROOT / candidate.name),
            "log_file": str(ACE_LOG_ROOT / f"{candidate.name}.log"),
        },
    }


def write_candidate_config(candidate: ACECandidate) -> Path:
    cfg = candidate_to_config(candidate)
    config_path = ACE_CONFIG_ROOT / f"{candidate.name}.json"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)
    return config_path

config_paths = {c.name: write_candidate_config(c) for c in ACE_CANDIDATES}
config_paths



## Launch script generation

This cell creates one shell script per candidate.

After you finalize the ACE training backend, you only need to set `ACE_TRAIN_CMD_TEMPLATE` and rerun this section.


In [ ]:

def build_train_command(candidate: ACECandidate, config_file: Path) -> str:
    if ACE_TRAIN_CMD_TEMPLATE is None:
        raise ValueError(
            "ACE_TRAIN_CMD_TEMPLATE is not set. Define the real training command first."
        )
    model_dir = ACE_MODEL_ROOT / candidate.name
    model_dir.mkdir(parents=True, exist_ok=True)
    log_file = ACE_LOG_ROOT / f"{candidate.name}.log"
    return ACE_TRAIN_CMD_TEMPLATE.format(
        config_file=config_file,
        log_file=log_file,
        model_dir=model_dir,
        train_file=TRAIN_SPLIT_FILE,
        valid_file=VALID_SPLIT_FILE,
        test_file=TEST_FILE,
        candidate=candidate.name,
    )


def write_launch_script(candidate: ACECandidate, config_file: Path) -> Path:
    script_path = ACE_SCRIPT_ROOT / f"launch_{candidate.name}.sh"
    model_dir = ACE_MODEL_ROOT / candidate.name
    model_dir.mkdir(parents=True, exist_ok=True)

    if ACE_TRAIN_CMD_TEMPLATE is None:
        command = f"# TODO: set ACE_TRAIN_CMD_TEMPLATE and regenerate scripts for {candidate.name}"
    else:
        command = build_train_command(candidate, config_file)

    text = f"""#!/usr/bin/env bash
set -euo pipefail

mkdir -p "{ACE_LOG_ROOT}"
mkdir -p "{model_dir}"

{command}
"""
    script_path.write_text(text, encoding="utf-8")
    script_path.chmod(0o755)
    return script_path

script_paths = {c.name: write_launch_script(c, config_paths[c.name]) for c in ACE_CANDIDATES}
script_paths



## Optional local launcher

Use this only after the backend command is finalized.

For cluster runs, you may prefer to submit the generated shell scripts manually.


In [ ]:

def run_candidate_script(candidate_name: str, dry_run: bool = True):
    script = ACE_SCRIPT_ROOT / f"launch_{candidate_name}.sh"
    if not script.exists():
        raise FileNotFoundError(script)

    cmd = ["bash", script.as_posix()]
    print("Command:", " ".join(cmd))
    if dry_run:
        return None
    return subprocess.run(cmd, check=True)

# Example dry run:
# run_candidate_script("ACE_A", dry_run=True)



## Candidate registry table

This table is useful for logging and for the paper workflow.


In [ ]:

registry_rows = []
for c in ACE_CANDIDATES:
    registry_rows.append({
        "candidate": c.name,
        "cutoff_A": c.cutoff,
        "basis_functions": c.basis_functions,
        "notes": c.notes,
        "config_file": str(config_paths[c.name]),
        "script_file": str(script_paths[c.name]),
        "model_dir": str(ACE_MODEL_ROOT / c.name),
        "log_file": str(ACE_LOG_ROOT / f"{c.name}.log"),
    })

registry_df = pd.DataFrame(registry_rows)
registry_df



## Metrics parsing helpers

These helpers are intentionally tolerant.

They try to read:
- a `metrics.json` file inside the candidate model directory,
- or selected patterns from the text log.

You can adapt the regexes later once the final ACE backend is fixed.


In [ ]:

FLOAT_RE = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"


def parse_metrics_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def parse_log_metrics(log_path: Path) -> Dict[str, Any]:
    if not log_path.exists():
        return {}
    text = log_path.read_text(encoding="utf-8", errors="ignore")
    out: Dict[str, Any] = {}

    patterns = {
        "train_loss": rf"train[_\s-]*loss[^0-9+-]*({FLOAT_RE})",
        "valid_loss": rf"val(?:id)?[_\s-]*loss[^0-9+-]*({FLOAT_RE})",
        "energy_rmse": rf"energy[_\s-]*rmse[^0-9+-]*({FLOAT_RE})",
        "force_rmse": rf"force[_\s-]*rmse[^0-9+-]*({FLOAT_RE})",
        "stress_rmse": rf"stress[_\s-]*rmse[^0-9+-]*({FLOAT_RE})",
    }

    for key, pat in patterns.items():
        matches = re.findall(pat, text, flags=re.IGNORECASE)
        if matches:
            try:
                out[key] = float(matches[-1])
            except Exception:
                pass
    return out


def collect_candidate_metrics(candidate_name: str) -> Dict[str, Any]:
    model_dir = ACE_MODEL_ROOT / candidate_name
    log_path = ACE_LOG_ROOT / f"{candidate_name}.log"
    metrics_path = model_dir / METRICS_FILENAME

    row = {
        "candidate": candidate_name,
        "model_dir": str(model_dir),
        "log_file": str(log_path),
        "metrics_json": str(metrics_path),
        "metrics_json_exists": metrics_path.exists(),
        "log_exists": log_path.exists(),
    }
    row.update(parse_metrics_json(metrics_path))
    for k, v in parse_log_metrics(log_path).items():
        row.setdefault(k, v)
    return row


In [ ]:

metrics_df = pd.DataFrame([collect_candidate_metrics(c.name) for c in ACE_CANDIDATES])
metrics_df



## Save ACE candidate summary

This CSV will be useful later for:
- quick comparison,
- integration with Notebook 05,
- paper tables.


In [ ]:

summary_df = registry_df.merge(metrics_df, on="candidate", how="left")
summary_df.to_csv(ACE_SUMMARY_FILE, index=False)
print("Wrote:", ACE_SUMMARY_FILE)
summary_df



## Preliminary screening rule

At this stage, only discard candidates that are clearly unsuitable, for example:
- training failed,
- export failed,
- logs show divergence,
- metrics are catastrophically worse,
- runtime is completely impractical.

Do **not** make the final scientific choice here.
The real production ACE model should be selected later using **Notebook 05**.


In [ ]:

def preliminary_status(row: pd.Series) -> str:
    if not bool(row.get("log_exists", False)):
        return "not_run"
    if row.get("valid_loss") is None and row.get("force_rmse") is None and row.get("energy_rmse") is None:
        return "inspect_manually"
    return "viable"

if len(summary_df) > 0:
    screening_df = summary_df.copy()
    screening_df["preliminary_status"] = screening_df.apply(preliminary_status, axis=1)
    screening_df
else:
    print("No summary rows available yet.")



## Notes for integration with Notebook 05

After one or more ACE candidates are trained and exported, register the viable model(s)
in `cuzr_setup_updated.py` or in a dedicated ACE helper registry.

Suggested workflow:
1. keep candidate names stable: `ACE_A`, `ACE_B`, `ACE_C`, `ACE_D`
2. export the final inference-ready ACE files into a predictable location
3. register each candidate for LAMMPS / pyiron use
4. run Notebook 05 on:
   - all viable ACE candidates,
   - MACE candidates,
   - EAM baselines
5. choose the final production ACE model there


In [ ]:

# Optional project note written to disk
notes_path = ACE_RUN_ROOT / "README_03c_notes.txt"
notes_text = """
03c purpose:
- train four ACE candidates on the Zenodo Cu-Zr dataset
- keep train/valid split fixed
- do not choose the final production model here
- pass viable candidates to Notebook 05 for physics-oriented validation
""".strip()
notes_path.write_text(notes_text, encoding="utf-8")
print("Wrote:", notes_path)



## Final reminder

This notebook is intentionally designed to be:
- symmetrical with the 4-model MACE study,
- conservative about overfitting,
- easy to connect to Notebook 05,
- easy to move onto the A100/B200 environment later.

Once the exact ACE backend is finalized, the main remaining edit is to set:

```python
ACE_TRAIN_CMD_TEMPLATE = "..."
```

Then regenerate launch scripts and run the candidates.
